# CeNN Attention Preservation Benchmark

This notebook answers one narrow question before changing the language-model architecture again:

**Can recurrent CeNN attention reproduce the pretrained Transformer's softmax attention when every other Transformer element is held fixed?**

The benchmark keeps **Q, K, V, RoPE, O projection, RMSNorm, residual path and MLP unchanged**. Only `softmax(QK^T/sqrt(d))V` is compared with the recurrent CeNN state `phi(q)^T S / phi(q)^T z`.

It sweeps CeNN feature sizes **128, 256, 512, 1024, 2048, 4096**, measures element-level and distribution-level errors, and reports the recurrent-state/KV-cache break-even context length.

In [ ]:
import importlib, pathlib, subprocess, sys

REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/vtavakkoli/TinyCeNN-LM.git', str(REPO_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()
import tinycenn_lm

commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Repository ready:', REPO_DIR)
print('Commit:', commit)
print('tinycenn_lm:', tinycenn_lm.__file__)

## 1. Configure the preservation sweep
The default layers include easy, medium, hard and previously problematic layers. Two natural-text sequences are enough for the first diagnostic; increase to 4 after the pipeline looks correct.

In [ ]:
from pathlib import Path

FEATURE_DIMS = [128, 256, 512, 1024, 2048, 4096]
LAYERS = '0,6,14,18,20,23,29'
CONTEXT_LENGTH = 128
NUM_SEQUENCES = 2
OUTPUT_DIR = REPO_DIR / 'result' / 'cenn-attention-preservation'

print('Feature sweep:', FEATURE_DIMS)
print('Layers:', LAYERS)
print('Context:', CONTEXT_LENGTH)
print('Output:', OUTPUT_DIR)

## 2. Run the isolated attention benchmark
For each selected layer the script reconstructs the Transformer's exact causal attention from its real Q/K/V tensors, verifies that reconstruction against the actual teacher attention output, then replaces only the attention kernel with CeNN random features.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, str(REPO_DIR / 'scripts' / 'benchmark_cenn_attention_preservation.py'),
    '--base-model', 'HuggingFaceTB/SmolLM2-135M',
    '--context-length', str(CONTEXT_LENGTH),
    '--num-sequences', str(NUM_SEQUENCES),
    '--layers', LAYERS,
    '--feature-dims', ','.join(map(str, FEATURE_DIMS)),
    '--query-samples', '12',
    '--head-samples', '3',
    '--top-k', '5',
    '--output-dir', str(OUTPUT_DIR),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 3. Summary table
Key interpretation: `output_cosine -> 1`, `NMSE -> 0`, `partition_log_mae -> 0`, `attention_JS -> 0`, and `top-k overlap -> 1` are better.

In [ ]:
import json, pandas as pd
from IPython.display import display

report = json.loads((OUTPUT_DIR / 'attention_preservation_report.json').read_text())
summary = pd.DataFrame(report['summary_by_feature_dim'])
cols = [c for c in [
    'feature_dim', 'output_cosine', 'output_nmse', 'output_pearson',
    'partition_log_mae', 'attention_kl', 'attention_js', 'topk_overlap',
    'o_projection_cosine', 'residual_cosine', 'state_vs_kv_ratio',
    'break_even_tokens', 'cenn_state_mib_fp32'
] if c in summary.columns]
display(summary[cols].round(6))
print('Teacher reconstruction sanity:', report['teacher_reconstruction_sanity'])
print('Recommended minimum F under benchmark thresholds:', report['recommended_min_feature_dim_for_0_99_cos_and_0_02_nmse'])

## 4. Fidelity curves

In [ ]:
import matplotlib.pyplot as plt

x = summary['feature_dim']
fig = plt.figure(figsize=(8, 4.5))
plt.plot(x, summary['output_cosine'], marker='o')
plt.axhline(0.99, linestyle='--')
plt.xscale('log', base=2)
plt.ylim(0, 1.01)
plt.xlabel('CeNN feature dimension')
plt.ylabel('Attention-output cosine')
plt.title('CeNN preservation of Transformer attention output')
plt.grid(True, alpha=0.25)
plt.show()

fig = plt.figure(figsize=(8, 4.5))
plt.plot(x, summary['output_nmse'], marker='o', label='attention output NMSE')
plt.plot(x, summary['o_projection_nmse'], marker='o', label='after O projection')
plt.xscale('log', base=2)
plt.yscale('log')
plt.xlabel('CeNN feature dimension')
plt.ylabel('NMSE (lower is better)')
plt.title('Approximation error vs CeNN state size')
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

fig = plt.figure(figsize=(8, 4.5))
plt.plot(x, summary['partition_log_mae'], marker='o')
plt.xscale('log', base=2)
plt.yscale('log')
plt.xlabel('CeNN feature dimension')
plt.ylabel('|log Z_CeNN - log Z_softmax|')
plt.title('Does CeNN preserve the Transformer attention normalization?')
plt.grid(True, alpha=0.25)
plt.show()

## 5. Layer-by-layer comparison

In [ ]:
layer_df = pd.DataFrame(report['summary_by_layer_feature'])
cos_pivot = layer_df.pivot(index='layer', columns='feature_dim', values='output_cosine')
nmse_pivot = layer_df.pivot(index='layer', columns='feature_dim', values='output_nmse')
print('Attention-output cosine by layer')
display(cos_pivot.round(5))
print('Attention-output NMSE by layer')
display(nmse_pivot.round(5))

fig = plt.figure(figsize=(9, 5))
for layer in cos_pivot.index:
    plt.plot(cos_pivot.columns, cos_pivot.loc[layer], marker='o', label=f'layer {layer}')
plt.xscale('log', base=2)
plt.xlabel('CeNN feature dimension')
plt.ylabel('Attention-output cosine')
plt.title('Which Transformer layers require a larger CeNN state?')
plt.legend(ncol=2)
plt.grid(True, alpha=0.25)
plt.show()

## 6. Transformer-element equivalence matrix

In [ ]:
equiv = pd.DataFrame(
    [{'Transformer element': k, 'What this benchmark does': v} for k, v in report['transformer_element_equivalence'].items()]
)
display(equiv)

## 7. State-size tradeoff
A larger CeNN state can improve fidelity, but for short contexts it can be larger than a normal KV cache. The important quantity is the **break-even context length**: beyond this length the recurrent state stays constant while Transformer KV memory keeps growing.

In [ ]:
mem_cols = [c for c in ['feature_dim','state_vs_kv_ratio','break_even_tokens','cenn_state_mib_fp32'] if c in summary.columns]
display(summary[mem_cols].round(4))

fig = plt.figure(figsize=(8, 4.5))
plt.plot(summary['feature_dim'], summary['break_even_tokens'], marker='o')
plt.xscale('log', base=2)
plt.yscale('log', base=2)
plt.xlabel('CeNN feature dimension')
plt.ylabel('Break-even context length (tokens)')
plt.title('When does constant CeNN state become smaller than Transformer KV cache?')
plt.grid(True, alpha=0.25)
plt.show()

## 8. Optional: verify the selected size on all 30 layers
Run this only after inspecting the sweep above. It uses the smallest feature size that achieved average cosine >= 0.99 and NMSE <= 0.02; if none did, it uses 4096.

In [ ]:
RUN_ALL_LAYERS = False
BEST_F = report['recommended_min_feature_dim_for_0_99_cos_and_0_02_nmse'] or 4096
ALL_DIR = REPO_DIR / 'result' / f'cenn-attention-all-layers-F{BEST_F}'

if RUN_ALL_LAYERS:
    all_cmd = [
        sys.executable, str(REPO_DIR / 'scripts' / 'benchmark_cenn_attention_preservation.py'),
        '--base-model', 'HuggingFaceTB/SmolLM2-135M',
        '--context-length', str(CONTEXT_LENGTH),
        '--num-sequences', str(NUM_SEQUENCES),
        '--layers', 'all',
        '--feature-dims', str(BEST_F),
        '--output-dir', str(ALL_DIR),
    ]
    subprocess.run(all_cmd, check=True)
    all_report = json.loads((ALL_DIR / 'attention_preservation_report.json').read_text())
    display(pd.DataFrame(all_report['summary_by_layer_feature']).round(6))
else:
    print('Set RUN_ALL_LAYERS = True when you want the full 30-layer confirmation. BEST_F =', BEST_F)

## How to interpret the result

A useful first target is **attention-output cosine >= 0.99** and **NMSE <= 0.02** across the difficult layers, with a small normalization error and high top-k overlap. If fidelity keeps improving as 1024 -> 2048 -> 4096, CeNN capacity was a primary limitation. If it saturates well below those targets, the issue is not merely state size and we should change the feature/kernel formulation before another LM training run.